# Smooth data from interpolated csv

In [2]:
import os
import glob
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
import matplotlib.pyplot as plt

#save smooth csv

In [1]:
#Input dir- file containing interpol csv, output dir- file where smooth data should be saved
INPUT_DIR  = r"./../../Tanvi-files/xyz_Interpolated/"
OUTPUT_DIR = r"./../../Tanvi-files/xyz_Smooth"
os.makedirs(OUTPUT_DIR, exist_ok=True)


body_points = {"pt1", "pt2", "pt3", "pt4", "pt7", "center"}  # body + wing bases + center
wing_tips   = {"pt5", "pt6"}                                # wing tips


# Nan-safe Savitzky-Golay 
def savgol_nan_safe(data, window, poly=3):
    data = data.copy()
    smooth = data.copy()
    isnan = np.isnan(data)
    s = None

    for i in range(len(data)):
        if not isnan[i] and s is None:
            s = i

        elif (isnan[i] or i == len(data) - 1) and s is not None:
            e = i if isnan[i] else i + 1
            seg = data[s:e]

            if len(seg) >= window:
                smooth[s:e] = savgol_filter(seg, window, poly)
            else:
                smooth[s:e] = seg

            s = None

    return smooth


# Find interpolated files

csv_files = glob.glob(os.path.join(INPUT_DIR, "*_INTERP.csv"))
if not csv_files:
    raise FileNotFoundError("No INTERP CSV files found.")


# Smooth

for path in csv_files:
    fname = os.path.basename(path)
    print("Smoothing:", fname)

    df = pd.read_csv(path)

    smooth_df = pd.DataFrame()
    smooth_df["frame"] = df["frame"]   # frame preserved

 
    # Smooth all columns except frame 
    
    for col in df.columns[1:]:
        pt = col.split("_")[0]

        if pt in body_points:
            window = 15
        elif pt in wing_tips:
            window = 7
        else:
            continue

        smooth_df[col] = savgol_nan_safe(df[col].values, window)

 
    # Save

    out_name = fname.replace("_INTERP.csv", "_SMOOTH.csv")
    smooth_df.to_csv(os.path.join(OUTPUT_DIR, out_name), index=False)

print("\n Smoothed CSVs (center included, not recomputed) saved.")

NameError: name 'os' is not defined